# Niural support agent - v1

Groq `openai/gpt-oss-120b`, six account tools, and the supplied documentation.
Use **Streaming chat** for conversation, or run one selected benchmark question. Judging is optional.
No automatic benchmark loop, retries, manifests, or tracing.


## 1. Setup
Select the **Conda base** kernel (`C:/Users/Nirvana/miniconda3/python.exe`). `GROQ_API_KEY` is loaded from `.env`.
This cell creates the agent without making a model request.


In [ ]:
from pathlib import Path
import json
import sys
from IPython.display import display, Markdown

repo = next(p for p in [Path.cwd(), *Path.cwd().parents]
            if (p / "mock_api/data").is_dir() and (p / "niural_support_agent/config.py").is_file())
if str(repo) not in sys.path:
    sys.path.insert(0, str(repo))

from niural_support_agent.config import Config, BENCHMARK_DIR
from mock_api import MockAPI, APIError
from niural_support_agent.agent import SupportAgent
from niural_support_agent.evaluate import evaluate_one

config = Config.load()
agent = SupportAgent(config)
display({"provider": config.provider, "model": config.model})


## 2. Mock API and token auth
`mock_api/` contains the client, auth, and Pydantic data models. Any nonempty
mock token is accepted; a missing token raises a 401 API error. This token is separate from the Groq key.


In [ ]:
api = MockAPI(token=config.mock_token)
employees = api.list_employees()
display({"employee_records": len(employees.data), "source": employees.source.model_dump(mode="json")})

try:
    MockAPI(token="").list_vendors()
except APIError as error:
    display(error.as_dict())

display([tool.name for tool in agent.agent.tools])


## 3. Streaming chat
Run this cell, then type a message. Each message starts one agent turn; text appears as it arrives.
`/reset` clears this chat, `/quit` exits. No judge calls are made.


In [ ]:
chat_session = "notebook-chat"
print("Chat ready. /reset clears history; /quit exits.")

while True:
    try:
        question = input("You: ").strip()
    except EOFError:
        break
    if question.lower() == "/quit":
        break
    if question.lower() == "/reset":
        agent.reset(chat_session)
        print("Conversation cleared.")
        continue
    if not question:
        continue

    print("Assistant: ", end="", flush=True)
    chunks = []
    def show_text(text):
        chunks.append(text)
        print(text, end="", flush=True)

    response = await agent.ask(chat_session, question, on_text=show_text)
    if response.error:
        print("\n" + (response.error.get("detail") or response.error["message"]))
    elif not chunks:
        print(response.answer, end="")
    print("\n")


## 4. Select one benchmark question
Change `case_id` to any integer from 1 to 8. Reference facts are used only by the optional judge.


In [ ]:
cases = json.loads((BENCHMARK_DIR / "cases.json").read_text(encoding="utf-8"))
case_id = 1
case = next(row for row in cases if row["id"] == case_id)
print(case["question"])


## 5. Generate one benchmark answer
Set `RUN_ANSWER = True` when ready. One question may require several model/tool calls
(up to four model turns). A rate-limit error stops the run; there are no automatic retries.


In [ ]:
RUN_ANSWER = False
result = None
if RUN_ANSWER:
    result = await evaluate_one(agent, case)
    display(Markdown(result["answer"]) if result["answer"] else result["error"])
    display(result["usage"])
else:
    print("Answer generation is off. Set RUN_ANSWER = True to run this question.")


## 6. Optional correctness judge
This makes an additional model request. Leave it off to save quota.
The answer is compared with the verified reference facts using the shared rubric.


In [ ]:
RUN_JUDGE = False
if RUN_JUDGE and result and result.get("answer") and result["id"] == case["id"]:
    verdict, usage = await agent.judge(
        case["question"], result["context"], case["facts"], result["answer"])
    display(verdict)
    display(usage)
else:
    print("Judge is off. Enable it explicitly after generating an answer.")


## 7. Notes for v2

- What worked:
- What failed:
- One proposed change:
- Retain / rollback / inconclusive:

Results remain in notebook outputs. Nothing is automatically saved to run folders.
